# Configuration

In [79]:

import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# Konfigurasi URL API
# Ganti URL berikut dengan URL ngrok yang dihasilkan dari script 004_public_endpoint.py

API_URL = "https://xxxx-xxx-xxx-xxx-xxx.ngrok-free.app"  # Ganti dengan URL ngrok aktif

def tampilkan_status(response):
    """Fungsi untuk menampilkan status response dari API"""
    if response.status_code >= 200 and response.status_code < 300:
        print(f"Sukses: Status code {response.status_code}")
    else:
        print(f"Error: Status code {response.status_code}")
        print(f"Pesan error: {response.json().get('detail', 'Tidak ada detail')}")
        
# BENAR: Membuat response dahulu melalui request
response = requests.get(API_URL)
tampilkan_status(response)  # Kemudian menggunakannya



# 1. Menambahkan Customer Baru

In [36]:

def tambah_customer(nama, umur, jenis_kelamin, total_revenue):
    """
    Fungsi untuk menambahkan customer baru
    
    Parameters:
    nama (str): Nama customer
    umur (int): Umur customer
    jenis_kelamin (str): Jenis kelamin (laki-laki atau perempuan)
    total_revenue (float): Total pendapatan dari customer
    
    Returns:
    dict: Data customer yang berhasil ditambahkan
    """
    url = f"{API_URL}/customers/"
    
    data = {
        "nama": nama,
        "umur": umur,
        "jenis_kelamin": jenis_kelamin,
        "total_revenue": total_revenue
    }
    
    response = requests.post(url, json=data)
    tampilkan_status(response)
    
    if response.status_code == 201:
        print("Customer berhasil ditambahkan!")
        return response.json()
    else:
        return None


In [ ]:
# 1. Menambahkan Customer
tambah_customer("Budi Santoso", 35, "laki-laki", 500_000)
tambah_customer("Siti Aminah", 28, "perempuan", 800_000)
tambah_customer("Adam Sidiq", 42, "laki-laki", 7_500_000)
tambah_customer("Ratna Sari", 31, "perempuan", 2_100_000)
tambah_customer("Agus Salim", 45, "laki-laki", 9_800_000)

# 2. Melihat Daftar Customer

In [ ]:
def lihat_daftar_customer():
    """
    Fungsi untuk melihat daftar nama semua customer
    
    Returns:
    list: Daftar nama customer
    """
    url = f"{API_URL}/customers/"
    
    response = requests.get(url)
    tampilkan_status(response)
    
    if response.status_code == 200:
        daftar_nama = response.json()
        print("\n📋 Daftar Customer:")
        for i, nama in enumerate(daftar_nama, 1):
            print(f"{i}. {nama}")
        return daftar_nama
    else:
        return []

In [ ]:
# 2. Melihat Daftar Customer
lihat_daftar_customer()

# 3. Lihat Detail Customer berdasarkan Customer ID

In [49]:
def lihat_detail_customer(customer_id):
    """
    Fungsi untuk melihat detail customer berdasarkan ID
    
    Parameters:
    customer_id (int): ID customer
    
    Returns:
    dict: Data detail customer
    """
    url = f"{API_URL}/customers/{customer_id}"
    
    response = requests.get(url)
    tampilkan_status(response)
    
    if response.status_code == 200:
        data = response.json()
        print("\n👤 Detail Customer:")
        print(f"ID: {data['id']}")
        print(f"Nama: {data['nama']}")
        print(f"Umur: {data['umur']} tahun")
        print(f"Jenis Kelamin: {data['jenis_kelamin']}")
        print(f"Total Revenue: Rp {data['total_revenue']:,.2f}")
        return data
    else:
        return None

In [ ]:
# 3. Melihat Detail Customer (ganti ID dengan ID yang sesuai)
lihat_detail_customer(1)


# 4. Melakukan Prediksi untuk Satu Customer


In [56]:
def prediksi_churn(umur, jenis_kelamin, total_revenue):
    """
    Fungsi untuk memprediksi churn untuk satu customer
    
    Parameters:
    umur (int): Umur customer
    jenis_kelamin (str): Jenis kelamin (laki-laki atau perempuan)
    total_revenue (float): Total pendapatan dari customer
    
    Returns:
    dict: Hasil prediksi churn
    """
    url = f"{API_URL}/predict/churn"
    
    data = {
        "umur": umur,
        "jenis_kelamin": jenis_kelamin,
        "total_revenue": total_revenue
    }
    
    response = requests.post(url, json=data)
    tampilkan_status(response)
    
    if response.status_code == 200:
        hasil = response.json()
        print(f"\nHasil Prediksi:")
        print(f"Probabilitas Churn: {hasil['churn_probability']:.2%}")
        print(f"Prediksi Churn: {'Ya' if hasil['churn_prediction'] else 'Tidak'}")
        print(f"Pesan: {hasil['message']}")
        return hasil
    else:
        return None

In [ ]:
# 4. Melakukan Prediksi untuk Satu Customer
prediksi_churn(35, "laki-laki", 2_500_000)


# 5. Melakukan Prediksi Bulk untuk Semua Customer

In [76]:

# Fungsi Tambahan: Visualisasi Hasil Prediksi
def visualisasi_hasil_prediksi(df_results):
    """
    Fungsi untuk memvisualisasikan hasil prediksi
    
    Parameters:
    df_results (DataFrame): DataFrame hasil prediksi
    """
    plt.figure(figsize=(15, 10))
    
    # Plot 1: Probabilitas Churn untuk Setiap Customer
    plt.subplot(2, 2, 1)
    bar_plot = sns.barplot(x='customer_name', y='churn_probability', 
                           hue='churn_prediction', data=df_results)
    plt.title('Probabilitas Churn per Customer')
    plt.xticks(rotation=45, ha='right')
    plt.ylim(0, 1)
    plt.ylabel('Probabilitas Churn')
    plt.xlabel('Nama Customer')
    plt.legend(title='Prediksi Churn')
    
    # Plot 2: Distribusi Probabilitas Churn
    plt.subplot(2, 2, 2)
    sns.histplot(df_results['churn_probability'], bins=10, kde=True)
    plt.title('Distribusi Probabilitas Churn')
    plt.xlabel('Probabilitas Churn')
    plt.ylabel('Jumlah Customer')
    
    # Plot 3: Pie Chart Prediksi Churn
    plt.subplot(2, 2, 3)
    churn_counts = df_results['churn_prediction'].value_counts()
    labels = ['Tidak Churn', 'Churn']
    colors = ['#5cb85c', '#d9534f']
    
    plt.pie(churn_counts, labels=labels, autopct='%1.1f%%', 
            startangle=90, colors=colors, explode=[0, 0.1])
    plt.title('Proporsi Prediksi Churn')
    
    plt.tight_layout()
    plt.show()

In [77]:

def prediksi_bulk():
    """
    Fungsi untuk melakukan prediksi bulk terhadap semua customer yang tersimpan
    
    Returns:
    dict: Hasil prediksi untuk semua customer dan ringkasan
    """
    url = f"{API_URL}/predict/churn/all-customers"
    
    response = requests.get(url)
    tampilkan_status(response)
    
    if response.status_code == 200:
        hasil = response.json()
        
        # Tampilkan ringkasan
        summary = hasil["summary"]
        print("\n📊 RINGKASAN PREDIKSI CHURN:")
        print(f"Total Customer: {summary['total_customers']}")
        print(f"Jumlah Customer Berisiko Churn: {summary['predicted_churn_count']}")
        print(f"Tingkat Churn: {summary['churn_rate']:.2%}")
        print(f"Rata-rata Probabilitas Churn: {summary['avg_churn_probability']:.2%}")
        
        print("\n⚠️ Customer Berisiko Tinggi:")
        for name in summary["high_risk_customers"]:
            print(f"- {name}")
        
        # Visualisasi hasil
        if len(hasil["results"]) > 0:
            df_results = pd.DataFrame(hasil["results"])
            visualisasi_hasil_prediksi(df_results)
            
        return hasil
    else:
        return None

In [ ]:
# 5. Melakukan Prediksi Bulk
df_result = prediksi_bulk()

In [ ]:
df_result

In [ ]:
# Contoh Penggunaan
print("=" * 50)
print("CLIENT API PREDIKSI CUSTOMER CHURN")
print("=" * 50)
print("Kode ini digunakan untuk berinteraksi dengan API prediksi customer churn")
print("Pastikan API sudah berjalan dan URL sudah diatur dengan benar")
print("=" * 50)
